# centraal-dataframework 0.2 con Azure Synapse serverless

Este notebook muestra cómo escribir tareas del framework que usan **Synapse serverless como motor de cómputo**.

**Regla de oro:** la Function **orquesta** y Synapse **procesa**. La tarea envía SQL y recibe resultados
pequeños (conteos, métricas, metadata). Los datos grandes no viajan a la Function: los lee, cruza, agrega y
escribe Synapse.

| Quiero… | Uso |
|---|---|
| Un número (conteo, máximo, fecha) | `ctx.synapse.scalar(sql, *params)` |
| Pocas filas como diccionarios | `ctx.synapse.query(sql, *params)` |
| Pocas filas como DataFrame | `ctx.synapse.read_df(sql, *params)` (pandas) |
| Crear vistas, CETAS, DDL | `ctx.synapse.execute_file('sql/x.sql', **params)` o `execute(sql)` |
| Otra base del mismo workspace | `ctx.synapse_for('silver')` |

Documentos relacionados:

- [MOTOR_DE_COMPUTO.md](MOTOR_DE_COMPUTO.md): cuándo usar pandas o polars y qué cuesta cada opción (Synapse hoy, BigQuery a futuro).
- [synapse_integracion.ipynb](synapse_integracion.ipynb): verifica que el ambiente funciona (driver, firewall, permisos).
- [MIGRATION_0.1_to_0.2.md](MIGRATION_0.1_to_0.2.md): API `TaskContext`.

## 0. Preparación

Antes de este notebook, el ambiente debe pasar [synapse_integracion.ipynb](synapse_integracion.ipynb)
(pasos 1 a 6: extra `[synapse]`, *ODBC Driver 18*, `az login`, firewall y usuario en la base).

Se usa el mismo archivo `.env.synapse` en la raíz del repo (copia de [synapse.env.example](synapse.env.example)):

| Variable | Uso en este notebook |
|---|---|
| `SYNAPSE_WORKSPACE` / `SYNAPSE_DATABASE` | Conexión (obligatorias) |
| `SYNAPSE_TEST_TABLE` | Tabla o vista de ejemplo, `esquema.tabla` (secciones 2, 3, 5, 6 y 8) |
| `SYNAPSE_TEST_DDL_SCHEMA` | Esquema donde se puede crear y borrar una vista (sección 4) |
| `SYNAPSE_TEST_OTHER_DB` | Otra base (sección 5) |
| `AZURE_STORAGE_CONNECTION_STRING` | Solo para la tarea con `gx` (Great Expectations guarda resultados en el storage) |

Las secciones cuya variable está vacía se saltan.

> Las salidas muestran filas del ambiente. **Limpie las salidas antes de commitear** (*Clear All Outputs*).

In [ ]:
import logging
import os
import sys
import tempfile
from pathlib import Path

from dotenv import load_dotenv

# Raíz del repo (el notebook vive en docs/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT))

ENV_FILE = ROOT / '.env.synapse'
if ENV_FILE.exists():
    load_dotenv(ENV_FILE, override=True)

# Solo para el notebook: ejecutar tareas sin YAML. En la Function App las tareas viven en el YAML.
os.environ['TAREA_DEBE_CONFIGURACION'] = 'false'

from centraal_dataframework.runner import reset_runner
from centraal_dataframework.settings import clear_settings_cache

clear_settings_cache()
reset_runner()
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

if not (os.environ.get('SYNAPSE_WORKSPACE') or os.environ.get('SYNAPSE_SERVER')):
    raise RuntimeError('Defina SYNAPSE_WORKSPACE en .env.synapse (ver synapse_integracion.ipynb, Paso 6)')

TABLA = os.environ.get('SYNAPSE_TEST_TABLE', '').strip()
DDL_SCHEMA = os.environ.get('SYNAPSE_TEST_DDL_SCHEMA', '').strip()
OTRA_BASE = os.environ.get('SYNAPSE_TEST_OTHER_DB', '').strip()
HAY_STORAGE = bool(os.environ.get('AZURE_STORAGE_CONNECTION_STRING') or os.environ.get('STORAGE_ACCOUNT_NAME'))

print('workspace :', os.environ.get('SYNAPSE_WORKSPACE') or os.environ.get('SYNAPSE_SERVER'))
print('base      :', os.environ.get('SYNAPSE_DATABASE') or '-')
print('tabla     :', TABLA or '(vacía: se saltan las secciones que la usan)')
print('ddl schema:', DDL_SCHEMA or '(vacío: se salta la sección 4)')
print('otra base :', OTRA_BASE or '(vacía: se salta la sección 5)')
print('storage   :', 'sí' if HAY_STORAGE else 'no (se salta la tarea con gx)')

## 1. El cliente

Dentro de una tarea se usa `ctx.synapse`. Fuera de una tarea (como aquí, para explorar) se usa `get_synapse()`.

Con solo `SYNAPSE_WORKSPACE`, el servidor es `<workspace>-ondemand.sql.azuresynapse.net` (serverless).
La autenticación por defecto es la identidad: `az login` en desarrollo, Managed Identity en la Function.

In [ ]:
from centraal_dataframework.synapse import get_synapse, render

synapse = get_synapse()
print(synapse)  # password y credencial no se muestran
synapse.query(
    "SELECT SUSER_SNAME() AS usuario, DB_NAME() AS base, "
    "CAST(SERVERPROPERTY('EngineEdition') AS int) AS engine_edition"  # 11 = serverless
)

## 2. Leer: pedirle a Synapse la respuesta, no los datos

Dos formas de parametrizar, según qué se parametriza:

- **Valores** (fechas, ids, filtros): parámetros `?` del driver. Nunca concatene valores en el SQL.
- **Identificadores** (esquema, tabla, base, carpeta): `render(sql, nombre=valor)` con `{{nombre}}`.
  Solo acepta letras, números y `_` separados por `-`, `:` o `.`; cualquier otra cosa da `ValueError`
  antes de tocar Synapse.

Serverless cobra por datos procesados (mínimo 10 MB por consulta). Un `COUNT_BIG(*)` o un `GROUP BY` que
devuelve pocas filas cuesta lo mismo que traer la tabla, pero no mueve los datos a la Function ni los
carga en memoria.

In [ ]:
if TABLA:
    filas = synapse.scalar(render('SELECT COUNT_BIG(*) FROM {{tabla}}', tabla=TABLA))
    print(f'{TABLA}: {filas} filas')

    # Solo los nombres de columna: TOP 0 no devuelve filas
    columnas = synapse.read_df(render('SELECT TOP 0 * FROM {{tabla}}', tabla=TABLA)).columns
    print(list(columnas))
else:
    print('SALTADO: defina SYNAPSE_TEST_TABLE')

Plantilla típica de una lectura agregada (ajuste columnas a su tabla):

```python
resumen = ctx.synapse.query(
    "SELECT canal, COUNT_BIG(*) AS filas, SUM(valor) AS total "
    "FROM gold.ventas WHERE fecha = ? GROUP BY canal",
    ctx.params['fecha'],
)
```

Pida solo las columnas que usa: en Parquet/Delta, serverless lee solo esas columnas y cobra menos.

## 3. Tareas: declarar qué plataformas usa

Una tarea es una función decorada con `@task` que recibe `TaskContext`. Cada tarea **declara** las
plataformas que carga en su contexto con `uses`:

| Declaración | Acceso en la tarea |
|---|---|
| `uses=['synapse']` | `ctx.synapse`, `ctx.synapse_for('silver')` |
| `uses=['datalake']` | `ctx.datalake` |
| `uses=['gx']` | `ctx.gx` |
| `uses=['synapse:gold']` | `ctx.resource('synapse', 'gold')`: otra conexión, con variables `SYNAPSE_WORKSPACE__GOLD`, … |

Reglas:

- Usar algo no declarado **falla** (`RecursoNoDeclarado`), aunque esté configurado.
- Si lo declarado no está configurado, la tarea **no se ejecuta** (`RecursoNoConfigurado`).
  `python -m centraal_dataframework check-resources -m mi_paquete.tareas` lo revisa para todas las
  tareas sin ejecutarlas.
- Declarar no conecta: cada cliente se crea la primera vez que la tarea lo usa.

Los parámetros llegan en `ctx.params` (del YAML, de la cola o de la llamada). Aquí las tareas se llaman
directamente para probarlas; en la Function las ejecuta el runner.

In [ ]:
from centraal_dataframework import TaskContext, task


@task(uses=['synapse'])
def contar_tabla(ctx: TaskContext):
    """Cuenta filas de una tabla; devuelve el conteo (un número, no los datos)."""
    n = ctx.synapse.scalar(render('SELECT COUNT_BIG(*) FROM {{tabla}}', tabla=ctx.params['tabla']))
    ctx.logger.info('filas=%s', n)  # loguear métricas, nunca filas
    return n


if TABLA:
    print(contar_tabla(params={'tabla': TABLA}))
else:
    print('SALTADO: defina SYNAPSE_TEST_TABLE')

## 4. Transformar dentro de Synapse: archivos `.sql` con `execute_file`

Las transformaciones viven en archivos `.sql` versionados junto a las tareas. `execute_file`:

1. sustituye `{{param}}` con la lista blanca,
2. separa el archivo en lotes por las líneas `GO`,
3. ejecuta cada lote en autocommit (serverless no admite DDL dentro de transacciones).

Las vistas no cuestan nada al crearse; se paga cuando alguien las consulta. Para **materializar** un
resultado que se reutiliza (y no recalcularlo en cada lectura) se usa CETAS, que escribe Parquet en el lago:

```sql
-- sql/ventas_diarias.sql  (requiere EXTERNAL DATA SOURCE y FILE FORMAT creados en la base)
IF OBJECT_ID('{{esquema}}.ventas_diarias_{{lote}}') IS NOT NULL
    DROP EXTERNAL TABLE {{esquema}}.ventas_diarias_{{lote}};
GO
CREATE EXTERNAL TABLE {{esquema}}.ventas_diarias_{{lote}}
WITH (LOCATION = 'gold/ventas_diarias/lote={{lote}}/', DATA_SOURCE = lago, FILE_FORMAT = parquet)
AS SELECT fecha, canal, SUM(valor) AS total
   FROM silver.ventas WHERE fecha = '{{lote}}' GROUP BY fecha, canal;
```

`DROP EXTERNAL TABLE` no borra los archivos: si la carpeta `LOCATION` ya existe, CETAS falla. Use una
carpeta por lote (como arriba) o limpie la carpeta antes.

La celda siguiente crea una vista sobre `SYNAPSE_TEST_TABLE` en `SYNAPSE_TEST_DDL_SCHEMA`, la consulta y la borra.

In [ ]:
import uuid

SQL_DIR = Path(tempfile.mkdtemp())  # en un proyecto real: carpeta sql/ del repo
(SQL_DIR / 'vista_muestra.sql').write_text(
    "CREATE OR ALTER VIEW {{esquema}}.cdf_ejemplo_{{run}} AS\n"
    "SELECT TOP 100 * FROM {{tabla}}\n"
    "GO\n",
    encoding='utf-8',
)


@task(uses=['synapse'])
def publicar_vista(ctx: TaskContext):
    """Crea o actualiza una vista; el cómputo ocurre cuando alguien la consulta."""
    p = ctx.params
    ctx.synapse.execute_file(SQL_DIR / 'vista_muestra.sql', esquema=p['esquema'], tabla=p['tabla'], run=p['run'])
    vista = f"{p['esquema']}.cdf_ejemplo_{p['run']}"
    n = ctx.synapse.scalar(f'SELECT COUNT_BIG(*) FROM {vista}')
    ctx.logger.info('vista=%s filas=%s', vista, n)
    return vista


if TABLA and DDL_SCHEMA:
    vista = publicar_vista(params={'esquema': DDL_SCHEMA, 'tabla': TABLA, 'run': uuid.uuid4().hex[:8]})
    synapse.execute(f'DROP VIEW IF EXISTS {vista}')
    print('creada, consultada y borrada:', vista)
else:
    print('SALTADO: defina SYNAPSE_TEST_TABLE y SYNAPSE_TEST_DDL_SCHEMA')

## 5. Varias bases (bronze / silver / gold)

`ctx.synapse_for('base')` devuelve un cliente para otra base del mismo workspace con la misma credencial.
El cliente original no cambia.

In [ ]:
@task(uses=['synapse'])
def bases_de_la_tarea(ctx: TaskContext):
    """Consulta dos bases desde la misma tarea."""
    actual = ctx.synapse.scalar('SELECT DB_NAME()')
    otra = ctx.synapse_for(ctx.params['otra_base']).scalar('SELECT DB_NAME()')
    return actual, otra


if OTRA_BASE:
    print(bases_de_la_tarea(params={'otra_base': OTRA_BASE}))
else:
    print('SALTADO: defina SYNAPSE_TEST_OTHER_DB')

## 6. Calidad de datos: calcular en SQL, validar el resultado

Great Expectations valida DataFrames de pandas en la Function. Llevar una tabla grande a la Function
solo para validarla contradice la regla de oro. El patrón es:

1. Synapse calcula las **métricas** (filas, nulos, duplicados, rangos) en una consulta: una fila.
2. La tarea valida esa fila: con una excepción, o con GX (`uses=[..., 'gx']`) para tener reporte y
   notificación de calidad. Ejecutada por el runner, un `DQResult` fallido es un error de calidad
   (salvo `silenciar_dq` vigente en el YAML); llamada directa, como aquí, devuelve el `DQResult`.

`gx` necesita el storage (`AZURE_STORAGE_CONNECTION_STRING`) porque GX guarda ahí suites y reportes.

In [ ]:
def sql_metricas(tabla: str) -> str:
    """Métricas de calidad calculadas por Synapse (una sola fila)."""
    return render('SELECT COUNT_BIG(*) AS filas FROM {{tabla}}', tabla=tabla)
    # Ejemplo con columnas reales:
    # SELECT COUNT_BIG(*) AS filas,
    #        SUM(CASE WHEN id IS NULL THEN 1 ELSE 0 END) AS ids_nulos,
    #        COUNT_BIG(*) - COUNT_BIG(DISTINCT id) AS ids_duplicados
    # FROM {{tabla}} WHERE fecha = ?


@task(uses=['synapse'])
def dq_basica(ctx: TaskContext):
    """Validación sin GX: falla la tarea si la tabla está vacía."""
    m = ctx.synapse.query(sql_metricas(ctx.params['tabla']))[0]
    if m['filas'] == 0:
        raise ValueError(f"{ctx.params['tabla']} está vacía")
    return m


if TABLA:
    print(dq_basica(params={'tabla': TABLA}))
else:
    print('SALTADO: defina SYNAPSE_TEST_TABLE')

In [ ]:
if TABLA and HAY_STORAGE:
    import great_expectations as gx

    @task(uses=['synapse', 'gx'])
    def dq_con_gx(ctx: TaskContext):
        """GX sobre las métricas (1 fila), no sobre la tabla."""
        metricas = ctx.synapse.read_df(sql_metricas(ctx.params['tabla']))
        return ctx.gx.validate(
            metricas,
            'metricas',
            [gx.expectations.ExpectColumnValuesToBeBetween(column='filas', min_value=1)],
        )

    resultado = dq_con_gx(params={'tabla': TABLA})
    print('success:', resultado.success, '| reporte:', resultado.url)
else:
    print('SALTADO: requiere SYNAPSE_TEST_TABLE y storage configurado')

## 7. Programar y ejecutar con el runner

En la Function App las tareas se declaran en el YAML con sus parámetros por defecto y su programación:

```yaml
tareas:
  publicar_ventas_diarias:
    horas: '6'
    params: {esquema: gold}
  dq_ventas_diarias:
    horas: '6'
    depends_on: [publicar_ventas_diarias]
    params: {tabla: gold.ventas_diarias}
```

El runner mezcla los `params` del YAML con los de la llamada, registra historial y notifica errores.
Localmente se puede ejecutar igual que en la Function:

In [ ]:
from centraal_dataframework.runner import get_runner

runner = get_runner()
print('tareas registradas:', runner.list_tasks())

if TABLA:
    print(runner.run_task('contar_tabla', params={'tabla': TABLA}, notify=False))

## 8. pandas y polars: solo con resultados pequeños

`read_df` devuelve un DataFrame de pandas con **el resultado** de la consulta. Úselo cuando el resultado
ya es pequeño (métricas, un resumen para un correo, un archivo para un usuario de negocio). Si necesita
polars, conviértalo desde `query` (no agrega dependencias a la librería).

Cuándo conviene cada uno, y cuándo no, está en [MOTOR_DE_COMPUTO.md](MOTOR_DE_COMPUTO.md).

In [ ]:
if TABLA:
    muestra = synapse.read_df(render('SELECT TOP 10 * FROM {{tabla}}', tabla=TABLA))  # TOP en el SQL, no .head()
    display(muestra)  # noqa: F821

    try:
        import polars as pl

        print(pl.DataFrame(synapse.query(render('SELECT TOP 10 * FROM {{tabla}}', tabla=TABLA))))
    except ImportError:
        print('polars no está instalado (opcional)')
else:
    print('SALTADO: defina SYNAPSE_TEST_TABLE')

## 9. Registrar una plataforma nueva

`datalake`, `synapse` y `gx` se registran con la misma API pública que puede usar un proyecto cliente
(y que usará el core para BigQuery). Se registra un nombre, una *factory* y una verificación de
configuración que no se conecta:

```python
from centraal_dataframework import register_resource


def crear_api(settings, ctx):
    """Crea el cliente; settings es de la conexión por defecto o de la conexión con nombre."""
    return MiApiClient(os.environ[settings.env_name('MI_API_URL')])


def verificar_api(settings):
    """Problemas de configuración, sin conectarse."""
    nombre = settings.env_name('MI_API_URL')
    return [] if os.environ.get(nombre) else [f'Configure {nombre}.']


register_resource('mi_api', crear_api, check=verificar_api)


@task(uses=['mi_api'])
def sincronizar(ctx: TaskContext):
    ctx.resource('mi_api').get(...)
```

El registro debe ocurrir **antes** de importar las tareas que lo declaran: declarar una plataforma
no registrada falla al importar el módulo de tareas.

## Antipatrones

| En vez de… | Haga… | Por qué |
|---|---|---|
| `read_df('SELECT * FROM tabla')` y filtrar/agrupar en pandas | `WHERE` / `GROUP BY` en el SQL | Los datos viajan a la Function y se cargan en memoria |
| `ctx.datalake.read_csv(...)` de un archivo grande para transformarlo | Vista o CETAS en Synapse sobre el archivo | El cómputo se queda en Synapse |
| `.head(10)` después de leer | `TOP 10` en el SQL | Igual se lee y transfiere todo |
| `SELECT *` sobre Parquet | Solo las columnas necesarias | Serverless cobra por columnas leídas |
| f-strings con valores en el SQL | `?` para valores, `render` para identificadores | Inyección SQL |
| Validar con GX una tabla completa en la Function | Métricas en SQL + GX sobre el resultado | Memoria y tiempo de la Function |
| Loguear filas (`ctx.logger.info(df)`) | Loguear conteos y duraciones | Los logs no son un lugar para datos |

## Function App

```python
import azure.functions as func
from centraal_dataframework.blueprints import framework
from mi_paquete import tareas  # noqa: F401  (registra las @task)

app = func.FunctionApp()
app.register_functions(framework)
```

En los *app settings*: `SYNAPSE_WORKSPACE`, `SYNAPSE_DATABASE` y `SYNAPSE_AUTH=identity`. La identidad
administrada de la Function necesita usuario y permisos en cada base (como en el Paso 5 de
[synapse_integracion.ipynb](synapse_integracion.ipynb)).